# Dancing with the Rivers: ANUGA dam-release scenarios on GPU (all dams)

**MODELLED. Scenario exploration, not a forecast. 30 m terrain.** The volumes (5 / 20 / 100 Mm³, 12 h triangular
release) are *screening* values, not reported events. ANUGA (GPL) runs here, offline; nothing of it ships in the web app.

What this notebook does, on **Google Colab or Kaggle** (detected automatically):

1. checks the GPU (`nvidia-smi`);
2. installs the NVIDIA HPC SDK from NVIDIA's apt repository and builds **ANUGA 4.2.0 from source with `nvc` and
   `-Dgpu_offload=true`** (flags taken from ANUGA's own `meson_options.txt` and `tools/install_anuga_nvc.sh`), then caches
   the wheel plus the NVHPC runtime libraries so later sessions skip the ~30–40 min build;
3. reads the dam packages (`<dam>.zip`, made by `pipeline/v2/anuga_package.py`);
4. runs a small **CPU-vs-GPU consistency test** before any real run;
5. runs every (dam, scenario) of the shard, with progress, wall time and resume-if-done;
6. writes one results zip per run (each with its own `run.json`), so results from Colab and Kaggle merge without
   conflicts in `pipeline/v2/anuga_ingest.py`.

> **The GPU build is untested by us.** We verified the GPU code paths exist in ANUGA 4.2.0 (`set_gpu_offload`,
> `set_compute_mode('unified')`, GPU `Inlet_operator`) and ran the same runner on CPU in Docker. The consistency test in
> step 4 is the guard: if it fails, use the CPU fallback below.

## Platform notes

**Colab.** Runtime → Change runtime type → GPU (T4 is fine; L4/A100 faster). Upload the package zips to
`MyDrive/dwr_anuga/packages/`. Results go to `MyDrive/dwr_anuga/results/<dam>/<scenario>/`. The built wheel is cached in
`MyDrive/dwr_anuga/wheels/`. Free sessions disconnect after a few hours: just re-run all cells; finished runs are skipped.

**Kaggle.** Settings → Accelerator **GPU T4 ×2** (preferred) or P100; **Internet ON** (needs a phone-verified account) —
apt and pip need it. Add the package zips as a dataset named **`dwr-anuga-packages`** (Add data → Upload →
the zips); they appear in `/kaggle/input/dwr-anuga-packages/`. Results are written to `/kaggle/working/results/`: download
them from the notebook's **Output** tab (or "Save Version" → output). To skip the build next time: after the first
session, download `/kaggle/working/wheels/` from Output, upload it as a dataset named **`dwr-anuga-wheel`**, and attach it;
the build cell then finds it in `/kaggle/input/dwr-anuga-wheel/`. Kaggle sessions stop at 12 h and the weekly GPU quota
is ~30 h: give Kaggle the long runs (see the split helper). One ANUGA process uses one GPU; on T4 ×2 the loop runs two
scenarios at a time (one per GPU) when `PARALLEL_GPUS = True`.

**P100 (compute capability 6.0).** CUDA 13 dropped Pascal, so the build cell then installs the `cuda-multi` SDK and
builds with `-Dgpu_arch=cuda12.9,cc60`. This is the least tested path: prefer T4.

## CPU fallback (`--mode cpu`)

If the GPU build fails, or the consistency test does not pass, set `MODE = "cpu"` below: the same runner then uses
ANUGA's **'unified' compute mode on the CPU** (the C Runge–Kutta loop, OpenMP threads, no device), with the stock pip
wheel `anuga==4.2.0` (no SDK, no build). `--mode legacy` is the v1 behaviour (Python evolve loop).

Expected CPU time: v1 measured 33–48 min for 69,616 triangles over 48 h simulated on 4 threads. These domains are larger
(see `est_triangles` / `est_cpu_h` in each `scenarios.json`): roughly **2–10 h per scenario** on Colab's 2 vCPUs, so on CPU
run only the fine `s20_d12` + its coarse E4 pair per dam first.

In [ ]:
# ---- settings (edit) --------------------------------------------------------------------------
SHARD = "all"            # "all", or an explicit list, e.g. [("sidi-salem", "s100_d12"), ("el-aroussia", "s20_d12")]
MODE = "gpu"             # "gpu" | "cpu" (unified on CPU, stock wheel) | "legacy"
ANUGA_TAG = "4.2.0"
NVHPC_VERSION = "25-9"   # ANUGA's validated SDK (docker/Dockerfile.gpu.slim: nvhpc 25.9)
PARALLEL_GPUS = True     # Kaggle T4 x2: run two scenarios at once, one per GPU
FORCE_REBUILD = False
SKIP_CONSISTENCY_TEST = False

In [ ]:
# ---- platform + paths -------------------------------------------------------------------------
import os, sys, json, glob, shutil, subprocess, time, zipfile, pathlib
ON_KAGGLE = os.path.exists("/kaggle")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive/dwr_anuga")
    PKG_SRC = BASE / "packages"
    RESULTS = BASE / "results"
    WHEELS = BASE / "wheels"
    WHEEL_SRC = [WHEELS]
    WORK = pathlib.Path("/content/dwr")
elif ON_KAGGLE:
    PKG_SRC = pathlib.Path("/kaggle/input/dwr-anuga-packages")
    RESULTS = pathlib.Path("/kaggle/working/results")
    WHEELS = pathlib.Path("/kaggle/working/wheels")
    WHEEL_SRC = [pathlib.Path("/kaggle/input/dwr-anuga-wheel"), WHEELS]
    WORK = pathlib.Path("/kaggle/temp/dwr") if os.path.exists("/kaggle/temp") else pathlib.Path("/tmp/dwr")
else:
    BASE = pathlib.Path.cwd() / "dwr_anuga"
    PKG_SRC, RESULTS, WHEELS = BASE / "packages", BASE / "results", BASE / "wheels"
    WHEEL_SRC = [WHEELS]
    WORK = BASE / "work"
for p in (RESULTS, WHEELS, WORK):
    p.mkdir(parents=True, exist_ok=True)
PLATFORM = "colab" if ON_COLAB else "kaggle" if ON_KAGGLE else "local"
print(PLATFORM, "| packages:", PKG_SRC, "| results:", RESULTS, "| wheels:", WHEELS)
print(sorted(p.name for p in PKG_SRC.glob("*.zip")) if PKG_SRC.exists() else "PACKAGE FOLDER MISSING")

## (a) GPU check

In [ ]:
!nvidia-smi
q = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,driver_version", "--format=csv,noheader"],
                   capture_output=True, text=True)
GPUS = [l.split(", ") for l in q.stdout.strip().splitlines()] if q.returncode == 0 else []
print("GPUs:", GPUS)
if MODE == "gpu":
    assert GPUS, "No GPU visible: switch the runtime to GPU, or set MODE = 'cpu'"
    CC = GPUS[0][1].replace(".", "")
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    CUDA_DRIVER = smi.split("CUDA Version:")[1].split()[0] if "CUDA Version:" in smi else "12.0"
    # CUDA 13 dropped compute capability < 7.5 (P100 = 6.0); a 12.x driver also needs a 12.x build
    USE_CUDA12 = int(CC) < 75 or CUDA_DRIVER.startswith("12")
    GPU_ARCH = f"cuda12.9,cc{CC}" if USE_CUDA12 else f"cc{CC}"
    NVHPC_PKG = f"nvhpc-{NVHPC_VERSION}-cuda-multi" if USE_CUDA12 else f"nvhpc-{NVHPC_VERSION}"
    print(f"compute capability {CC}, driver CUDA {CUDA_DRIVER} -> -Dgpu_arch={GPU_ARCH}, apt package {NVHPC_PKG}")

## (b) Install the NVIDIA HPC SDK and build ANUGA 4.2.0 with GPU offload (cached)

Build command, from ANUGA's `tools/install_anuga_nvc.sh` / `docker/Dockerfile.gpu.slim`:
`CC=nvc CFLAGS=-tp=haswell pip wheel --no-build-isolation -Csetup-args=-Dgpu_offload=true -Csetup-args=-Dgpu_arch=<arch>`
(`-tp=haswell` keeps the cached wheel portable across Colab/Kaggle host CPUs). The SDK download is ~4.5 GB
(~9 GB for `cuda-multi`). The NVHPC redistributable runtime libraries (`REDIST/compilers/lib`, e.g. libnvomp) are cached
next to the wheel: ANUGA runs in a subprocess with them on `LD_LIBRARY_PATH`.

In [ ]:
PYTAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
KEY = f"anuga-{ANUGA_TAG}-gpu-{GPU_ARCH.replace(',', '_')}-{PYTAG}" if MODE == "gpu" else None
REDIST = pathlib.Path("/opt/nvhpc-redist")
RUN_ENV = dict(os.environ)

def find_cached():
    for d in WHEEL_SRC:
        whl = sorted(d.glob(f"{KEY}/anuga-*.whl")) if d.exists() else []
        if whl and (whl[0].parent / "nvhpc-redist.tar.gz").exists():
            return whl[0]
    return None

def sh(cmd):
    print("+", cmd)
    subprocess.run(cmd, shell=True, check=True)

if MODE == "gpu":
    cached = None if FORCE_REBUILD else find_cached()
    if cached is None:
        t0 = time.time()
        sh("curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | "
           "gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg")
        sh("echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] "
           "https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' > /etc/apt/sources.list.d/nvhpc.list")
        sh(f"apt-get update -y -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {NVHPC_PKG}")
        NVC = sorted(glob.glob(f"/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc"))[-1]
        print("nvc:", NVC, f"({(time.time() - t0) / 60:.0f} min)")
        sh(f"{sys.executable} -m pip install -q 'numpy>=2.0' cython 'meson>=1.4' meson-python ninja pybind11")
        src = WORK / "anuga_core"
        if not src.exists():
            sh(f"git clone --depth 1 --branch {ANUGA_TAG} https://github.com/anuga-community/anuga_core.git {src}")
        out = WHEELS / KEY
        out.mkdir(parents=True, exist_ok=True)
        sh(f"cd {src} && rm -rf build && CC={NVC} CFLAGS=-tp=haswell {sys.executable} -m pip wheel --no-build-isolation "
           f"--no-deps -w {out} -Csetup-args=-Dgpu_offload=true -Csetup-args=-Dgpu_arch={GPU_ARCH} .")
        lib = pathlib.Path(sorted(glob.glob("/opt/nvidia/hpc_sdk/Linux_x86_64/*/REDIST/compilers/lib"))[-1])
        sh(f"tar -C {lib} -czf {out}/nvhpc-redist.tar.gz .")
        cached = sorted(out.glob("anuga-*.whl"))[0]
        print(f"built + cached {cached} in {(time.time() - t0) / 60:.0f} min")
    else:
        print("using cached wheel", cached)
    REDIST.mkdir(parents=True, exist_ok=True)
    sh(f"tar -C {REDIST} -xzf {cached.parent}/nvhpc-redist.tar.gz")
    sh(f"{sys.executable} -m pip install -q --force-reinstall --no-deps {cached}")
    sh(f"{sys.executable} -m pip install -q scipy matplotlib netCDF4 dill meshpy pymetis xarray rasterio shapely pyproj")
    RUN_ENV["LD_LIBRARY_PATH"] = f"{REDIST}:" + RUN_ENV.get("LD_LIBRARY_PATH", "")
    RUN_ENV["OMP_NUM_THREADS"] = "1"
    chk = subprocess.run([sys.executable, "-c", "import anuga; print(anuga.__version__, anuga.gpu_offload_supported())"],
                         env=RUN_ENV, capture_output=True, text=True)
    print(chk.stdout, chk.stderr[-2000:])
    assert chk.stdout.strip().endswith("True"), "anuga.gpu_offload_supported() is False: GPU build failed (use MODE='cpu')"
else:
    sh(f"{sys.executable} -m pip install -q anuga=={ANUGA_TAG} rasterio shapely pyproj")

## (c) Packages and the runner

The runner below is `pipeline/v2/anuga_run.py`, embedded verbatim.

In [ ]:
%%writefile anuga_run.py
#!/usr/bin/env python3
"""V2 · Run ONE ANUGA dam-release scenario from a self-contained package (dam-agnostic refactor of
pipeline/05_anuga_scenarios.py). MODELLED: scenario exploration, not a forecast. 30 m terrain.

  python anuga_run.py --package <dir with domain.json> --scenario s20_d12 --mode gpu|cpu|legacy [--out DIR]
  python anuga_run.py --package <dir> --scenario custom --volume-m3 2e7 --duration-h 12 --max-area 2500 --finaltime-h 1

Needs only: anuga (4.2.0), numpy, scipy, rasterio. ANUGA is GPL and runs offline only; nothing of it ships in the web app.

Modes
  legacy  V1 behaviour: Python evolve loop, OpenMP CPU kernels (multiprocessor mode 1).
  cpu     'unified' compute mode (mode 2) on the CPU: the C Runge–Kutta loop, no device (CPU fallback for Colab/Kaggle).
  gpu     'unified' + anuga.set_gpu_offload(True, strict=True): OpenMP target offload. Only on a GPU build of ANUGA
          (nvc, -Dgpu_offload=true); fails loudly otherwise.
  Mode 2 skips Python forcing terms other than Manning friction and the degenerate-timestep protection. This set-up
  uses neither: friction is Manning only, forcing is one Inlet_operator (GPU-registered), boundaries are Reflective +
  Dirichlet (C-side); the run checks for NaN at every yield step (test E3).

Set-up (as V1): uniform mesh (or mesh_zones refined) from domain.json; elevation = package DEM bilinear at centroids,
D8 path cells lowered burn_m; Manning n from the package raster; Inlet_operator on the crest line; triangular release
(peak 0.3 D, Q_peak = 2V/D); Dirichlet stage 2 m below the lowest outflow bed on 'outflow', reflective walls; dry start;
DE0. Outputs in --out: results.npz, run.json, maxdepth_30m.tif, maxspeed_30m.tif, arrival_h_30m.tif, stats.json,
tests.json (E1 mass balance, E2 inflow, E3 stability, E7 arrival).
"""
from __future__ import annotations

import argparse
import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

WET_M = 0.1


def log(*a):
    print(*a, flush=True)


def hydrograph(V, D):
    Qp, tp = 2 * V / D, 0.3 * D
    return (lambda t: Qp * t / tp if t < tp else (Qp * (D - t) / (D - tp) if t < D else 0.0)), Qp


def read(p):
    import rasterio
    with rasterio.open(p) as s:
        return s.read(1).astype(np.float64), s.transform, s.crs


def gpu_name():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                              capture_output=True, text=True, timeout=20).stdout.strip() or None
    except Exception:  # noqa: BLE001
        return None


def set_mode(domain, anuga, mode):
    info = dict(mode=mode, gpu_offload=False, gpu_offload_supported=bool(getattr(anuga, "gpu_offload_supported",
                                                                                 lambda: False)()))
    if mode == "legacy":
        domain.set_multiprocessor_mode(1)
        return info
    if mode == "gpu":
        anuga.set_gpu_offload(True, strict=True)
    if hasattr(domain, "set_compute_mode"):
        domain.set_compute_mode("unified")
    else:
        domain.set_multiprocessor_mode(2)
    info["compute_mode"] = domain.get_compute_mode() if hasattr(domain, "get_compute_mode") else "mode2"
    info["gpu_offload"] = bool(getattr(anuga, "gpu_offload_enabled", lambda: False)())
    if mode == "gpu" and not info["gpu_offload"]:
        raise RuntimeError("gpu mode requested but anuga.gpu_offload_enabled() is False (not a GPU build?)")
    return info


def run(pkg: Path, sc: dict, mode: str, out: Path, threads: int, active_set: bool):
    import anuga
    import rasterio
    from rasterio import features
    from scipy.ndimage import map_coordinates
    from shapely import polygons

    S = json.loads((pkg / "domain.json").read_text("utf-8"))
    out.mkdir(parents=True, exist_ok=True)
    x0, y0 = S["origin"]
    loc = lambda pts: [[p[0] - x0, p[1] - y0] for p in pts]                 # noqa: E731
    bounding = loc(S["domain"])
    nseg = len(bounding)
    outflow = set(S["outflow_segments"])
    tags = {"walls": [i for i in range(nseg) if i not in outflow], "outflow": sorted(outflow)}
    a_fine = float(sc["max_area_m2"])
    uniform = bool(sc.get("uniform", True))
    a_else = float(sc.get("max_area_elsewhere_m2", a_fine * 8))
    t0 = time.time()
    domain = anuga.create_domain_from_regions(
        bounding, boundary_tags=tags, maximum_triangle_area=a_fine if uniform else a_else,
        interior_regions=[] if uniform else [[loc(p), a_fine] for p in S["mesh_zones"]],
        mesh_filename=str(out / "mesh.msh"), use_cache=False, verbose=False)
    domain.set_name(sc["id"])
    domain.set_datadir(str(out))
    domain.set_minimum_storable_height(0.01)
    domain.set_flow_algorithm("DE0")
    ntri = len(domain)
    log(f"mesh: {ntri} triangles (≤ {a_fine:g} m²{'' if uniform else f' in zones, ≤ {a_else:g} m² elsewhere'}) in "
        f"{time.time() - t0:.0f} s; areas min/median/max {domain.areas.min():.0f}/{np.median(domain.areas):.0f}/"
        f"{domain.areas.max():.0f} m²")

    z, T, crs = read(pkg / "dem_30m.tif")
    path = np.load(pkg / "path_cells.npy")
    z[path[:, 0], path[:, 1]] -= float(S["burn_m"])
    mann, _, _ = read(pkg / "manning_30m.tif")

    def rowcol(x, y):
        return (T.f - (np.asarray(y) + y0)) / T.a - 0.5, ((np.asarray(x) + x0) - T.c) / T.a - 0.5

    def elev(x, y):
        r, c = rowcol(x, y)
        return map_coordinates(z, [r, c], order=1, mode="nearest")

    def fric(x, y):
        r, c = rowcol(x, y)
        return mann[np.clip(np.rint(r).astype(int), 0, mann.shape[0] - 1), np.clip(np.rint(c).astype(int), 0,
                                                                                    mann.shape[1] - 1)]

    domain.set_quantity("elevation", function=elev, location="centroids")
    domain.set_quantity("friction", function=fric, location="centroids")
    domain.set_quantity("stage", expression="elevation")
    V, D = float(sc["volume_m3"]), float(sc["duration_h"]) * 3600.0
    Q, Qp = hydrograph(V, D)
    tt = np.linspace(0, D, 200001)
    V_analytic = float(np.trapezoid([Q(t) for t in tt], tt))
    anuga.Inlet_operator(domain, loc(S["inlet_line"]), Q=Q)
    bxy = np.array(bounding)
    seg_z = []
    for i in outflow:
        a, b = bxy[i], bxy[(i + 1) % nseg]
        s = np.linspace(0, 1, 20)
        seg_z.append(elev(a[0] + s * (b[0] - a[0]), a[1] + s * (b[1] - a[1])).min())
    zmin = float(min(seg_z))
    domain.set_boundary({"walls": anuga.Reflective_boundary(domain),
                         "outflow": anuga.Dirichlet_boundary([zmin - 2.0, 0.0, 0.0])})
    os.environ["OMP_NUM_THREADS"] = str(threads)
    minfo = set_mode(domain, anuga, mode)
    if active_set and mode != "legacy" and hasattr(domain, "set_use_active_set"):
        domain.set_use_active_set(True)
    cen = domain.get_centroid_coordinates()
    elev_c = domain.quantities["elevation"].centroid_values.copy()
    areas = domain.areas.copy()
    log(f"scenario {sc['id']}: V = {V / 1e6:g} Mm³, D = {D / 3600:g} h, Q_peak = {Qp:.0f} m³/s (scenario value); "
        f"mode {json.dumps(minfo)}; outflow stage {zmin - 2:.1f} m")
    ys = float(sc.get("yieldstep_s", S.get("yieldstep_s", 300)))
    ft = float(sc["finaltime_h"]) * 3600
    frame_dt = 1800.0
    maxd, maxs, arr = np.zeros(ntri), np.zeros(ntri), np.full(ntri, np.nan)
    frames, ftimes, series = [], [], []
    nan_seen = False
    wall0 = time.time()
    for t in domain.evolve(yieldstep=ys, finaltime=ft):
        st = domain.quantities["stage"].centroid_values
        xm = domain.quantities["xmomentum"].centroid_values
        ym = domain.quantities["ymomentum"].centroid_values
        dep = np.maximum(st - elev_c, 0)
        if not (np.isfinite(st).all() and np.isfinite(xm).all() and np.isfinite(ym).all()):
            nan_seen = True
        spd = np.where(dep > WET_M, np.hypot(xm, ym) / np.maximum(dep, 1e-6), 0.0)
        maxd = np.maximum(maxd, dep)
        maxs = np.maximum(maxs, spd)
        arr = np.where(np.isnan(arr) & (dep > WET_M), t, arr)
        vol = float(domain.get_water_volume())
        bfi = float(domain.get_boundary_flux_integral())
        fsi = float(domain.get_fractional_step_volume_integral())
        series.append([float(t), vol, bfi, fsi, float(Q(t)), float((dep > WET_M).dot(areas)), float(spd.max()),
                       float(dep.max())])
        if abs(t / frame_dt - round(t / frame_dt)) < 1e-6:
            frames.append(dep.astype(np.float32))
            ftimes.append(float(t))
        if abs(t / 3600 - round(t / 3600)) < 1e-6 or t >= ft:
            log(f"  t = {t / 3600:5.1f} h | wall {time.time() - wall0:6.0f} s | volume {vol / 1e6:8.3f} Mm³ | out "
                f"{-bfi / 1e6:8.3f} | in {fsi / 1e6:8.3f} | wet {series[-1][5] / 1e6:7.2f} km² | max speed "
                f"{spd.max():.1f} m/s")
        if nan_seen:
            log("  NaN in the state: stopping (E3 fails)")
            break
    wall = time.time() - wall0
    nodes, tri = domain.nodes + np.array([x0, y0]), domain.triangles
    np.savez_compressed(out / "results.npz", centroids=cen + np.array([x0, y0]), nodes=nodes, triangles=tri,
                        areas=areas, elevation=elev_c, max_depth=maxd.astype(np.float32),
                        max_speed=maxs.astype(np.float32), arrival_s=arr.astype(np.float32),
                        frames=np.array(frames, dtype=np.float16), frame_t=np.array(ftimes), series=np.array(series))
    s = np.array(series)
    meta = dict(id=sc["id"], dam_id=S["dam_id"], dem=S["dem"], volume_m3=V, duration_s=D, Q_peak_m3s=Qp,
                V_analytic_m3=V_analytic, triangles=int(ntri), uniform=uniform, max_area_m2=a_fine,
                max_area_elsewhere_m2=None if uniform else a_else, wall_s=wall, finaltime_s=ft,
                simulated_s=float(s[-1, 0]), yieldstep_s=ys, nan_seen=bool(nan_seen), final_volume_m3=float(s[-1, 1]),
                boundary_flux_integral_m3=float(s[-1, 2]), inflow_applied_m3=float(s[-1, 3]),
                domain_km2=S["domain_km2"], outflow_stage_m=zmin - 2.0, anuga_version=anuga.__version__,
                burn_m=S["burn_m"], threads=threads, active_set=bool(active_set and mode != "legacy"),
                active_set_stats=(list(domain.get_active_set_stats()) if hasattr(domain, "get_active_set_stats")
                                  and mode != "legacy" else None),
                role=sc.get("role", "scenario"), pair_of=sc.get("pair_of"),
                platform=dict(host=platform.node(), python=platform.python_version(), gpu=gpu_name(),
                              kaggle=os.path.exists("/kaggle"), colab="google.colab" in sys.modules),
                **minfo)
    (out / "run.json").write_text(json.dumps(meta, indent=1), "utf-8")
    # rasters (30 m, package grid)
    H, W = z.shape
    idx = features.rasterize(((g, i + 1) for i, g in enumerate(polygons(nodes[tri]))), out_shape=(H, W),
                             transform=T, dtype="int32") - 1
    inside = idx >= 0

    def to30(v, fill):
        o = np.full((H, W), fill, np.float32)
        o[inside] = v[idx[inside]]
        return o
    md, ms = to30(maxd.astype(np.float32), 0), to30(maxs.astype(np.float32), 0)
    ar = to30((arr / 3600.0).astype(np.float32), np.nan)
    ar[md <= WET_M] = np.nan
    prof = dict(driver="GTiff", height=H, width=W, count=1, dtype="float32", crs=crs, transform=T, compress="deflate",
                predictor=3, tiled=True)
    for nm, a in (("maxdepth_30m", md), ("maxspeed_30m", ms), ("arrival_h_30m", ar)):
        with rasterio.open(out / f"{nm}.tif", "w", nodata=np.nan if nm.startswith("arrival") else None, **prof) as o:
            o.write(a, 1)
    # tests (V1 definitions)
    t_, vol, bfi, fsi = s[:, 0], s[:, 1], s[:, 2], s[:, 3]
    mb = (vol - (fsi + bfi)) / V
    tests = {}
    tests["E1"] = dict(mass_balance_err=float(abs(mb[-1])), max_abs_during_run=float(np.abs(mb).max()),
                       inflow_applied_m3=float(fsi[-1]), out_through_boundary_m3=float(-bfi[-1]),
                       final_volume_in_domain_m3=float(vol[-1]), passed=bool(abs(mb[-1]) < 0.01))
    full = t_[-1] >= D - 1
    tests["E2"] = dict(V_scenario_m3=V, V_hydrograph_integral_m3=V_analytic, err_analytic=float(abs(V_analytic - V) / V),
                       inflow_applied_by_anuga_m3=float(fsi[-1]), err_applied=float(abs(fsi[-1] - V) / V),
                       inflow_expected_by_t_end_m3=float(np.trapezoid([Q(x) for x in np.linspace(0, t_[-1], 20001)],
                                                                      np.linspace(0, t_[-1], 20001))),
                       full_release_simulated=bool(full))
    exp = tests["E2"]["inflow_expected_by_t_end_m3"]
    tests["E2"]["err_vs_expected_by_t_end"] = float(abs(fsi[-1] - exp) / max(exp, 1))
    tests["E2"]["passed"] = bool(tests["E2"]["err_analytic"] < 1e-3 and tests["E2"]["err_vs_expected_by_t_end"] < 1e-3)
    wet_t = maxd > WET_M
    tests["E3"] = dict(nan_seen=bool(nan_seen), max_speed_m_s=float(maxs.max()),
                       p99_speed_m_s=float(np.percentile(maxs[wet_t], 99)) if wet_t.any() else 0.0,
                       max_depth_m=float(maxd.max()), passed=bool((not nan_seen) and maxs.max() < 15))
    cxy = cen + np.array([x0, y0])
    arrival = {}
    for m in S["landmarks"]:
        d = np.hypot(cxy[:, 0] - m["xy"][0], cxy[:, 1] - m["xy"][1])
        near = (d < 300) & np.isfinite(arr)
        arrival[m["name"]] = round(float(np.nanmin(arr[near]) / 3600), 2) if near.any() else None
    reached = [(m["dist_km"], arrival[m["name"]]) for m in S["landmarks"] if arrival[m["name"]] is not None]
    # E7 fails (not "vacuously passes") when no landmark was reached, e.g. a short smoke run
    mono = all(b[1] >= a[1] - 0.05 for a, b in zip(sorted(reached), sorted(reached)[1:]))
    tests["E7"] = dict(arrival_h=arrival, monotonic_downstream=bool(mono),
                       first_outflow_h=float(t_[np.argmax(-bfi > 0.001 * V)] / 3600) if (-bfi > 0.001 * V).any() else None,
                       note="no reported event for these screening volumes: plausibility = arrival grows downstream and "
                            "mean front speed 0.1–5 m/s",
                       mean_front_speed_m_s={k: round(m["dist_km"] * 1000 / (arrival[m["name"]] * 3600), 2)
                                             for m in S["landmarks"] for k in [m["name"]]
                                             if arrival[m["name"]] not in (None, 0)},
                       passed=bool(reached and mono and all(0.1 <= v <= 5 for v in
                                                [m["dist_km"] * 1000 / (arrival[m["name"]] * 3600)
                                                 for m in S["landmarks"] if arrival[m["name"]] not in (None, 0)])))
    (out / "tests.json").write_text(json.dumps(tests, indent=1, ensure_ascii=False), "utf-8")
    stats = dict(mass_balance_err=float(abs(mb[-1])), peak_Q_m3s=float(Qp),
                 inundated_km2=round(float(areas[wet_t].sum() / 1e6), 3), arrival_h={k: v for k, v in arrival.items()
                                                                                    if v is not None},
                 max_depth_m={})
    (out / "stats.json").write_text(json.dumps(stats, indent=1, ensure_ascii=False), "utf-8")
    for k, v in tests.items():
        log(k, json.dumps(v, ensure_ascii=False))
    log(f"done in {wall / 60:.1f} min wall, {ntri} triangles; results → {out}")
    return meta, tests


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--package", type=Path, required=True)
    ap.add_argument("--scenario", required=True, help="id from scenarios.json, or a new id with --volume-m3")
    ap.add_argument("--volume-m3", type=float)
    ap.add_argument("--duration-h", type=float, default=None)
    ap.add_argument("--max-area", type=float, default=None, help="override max triangle area (m²)")
    ap.add_argument("--finaltime-h", type=float, default=None)
    ap.add_argument("--mode", choices=["gpu", "cpu", "legacy"], default="cpu")
    ap.add_argument("--threads", type=int, default=None, help="OpenMP threads (default: 1 for gpu, 4 otherwise)")
    ap.add_argument("--no-active-set", action="store_true")
    ap.add_argument("--out", type=Path, default=None, help="default <package>/../results/<dam>/<scenario>")
    a = ap.parse_args()
    S = json.loads((a.package / "domain.json").read_text("utf-8"))
    scs = {s["id"]: s for s in json.loads((a.package / "scenarios.json").read_text("utf-8"))["scenarios"]}
    sc = dict(scs.get(a.scenario) or dict(id=a.scenario, volume_m3=None, duration_h=12.0, max_area_m2=2500.0,
                                          uniform=True, finaltime_h=24.0, role="custom"))
    if a.volume_m3 is not None:
        sc["volume_m3"] = a.volume_m3
    for k, v in (("duration_h", a.duration_h), ("max_area_m2", a.max_area), ("finaltime_h", a.finaltime_h)):
        if v is not None:
            sc[k] = v
    if not sc.get("volume_m3"):
        raise SystemExit("unknown scenario: give --volume-m3")
    sc["id"] = a.scenario
    out = a.out or a.package.parent / "results" / S["dam_id"] / a.scenario
    run(a.package, sc, a.mode, out, a.threads or (1 if a.mode == "gpu" else 4), not a.no_active_set)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
PKGS = WORK / "packages"
PKGS.mkdir(exist_ok=True)
for z in sorted(PKG_SRC.glob("*.zip")):
    if not (PKGS / z.stem / "domain.json").exists():
        with zipfile.ZipFile(z) as zf:
            zf.extractall(PKGS)
DAMS = sorted(p.parent.name for p in PKGS.glob("*/scenarios.json"))
ALL = [(d, s) for d in DAMS for s in json.loads((PKGS / d / "scenarios.json").read_text())["scenarios"]]
for d, s in ALL:
    print(f"{d:20s} {s['id']:16s} V={s['volume_m3'] / 1e6:5g} Mm³  ≤{s['max_area_m2']:6g} m²  ~{s['est_triangles']:7d} tri  "
          f"~{s['est_cpu_h']:5.1f} h CPU (est.)  {s['finaltime_h']:g} h simulated")

### Split helper (Colab vs Kaggle)

Greedy split by estimated cost (triangles × simulated time / √area, weighted up for larger volumes, which wet more of
the domain): Kaggle (12 h sessions, 2 GPUs) gets the largest runs, Colab the rest. Copy a printed list into `SHARD`.

In [ ]:
def cost(s):
    import math
    return s["est_cpu_h"] * (1 + 0.3 * math.log10(max(s["volume_m3"], 5e6) / 5e6))

def suggest_split(share_kaggle=0.6):
    items = sorted(ALL, key=lambda x: -cost(x[1]))
    tot = sum(cost(s) for _, s in items)
    kag, col, ck = [], [], 0.0
    for d, s in items:
        if ck < share_kaggle * tot:
            kag.append((d, s["id"])); ck += cost(s)
        else:
            col.append((d, s["id"]))
    print(f"KAGGLE ({ck / tot * 100:.0f} % of est. cost):", kag)
    print(f"COLAB  ({100 - ck / tot * 100:.0f} %):", col)
    return kag, col

KAGGLE_SHARD, COLAB_SHARD = suggest_split()

## (d) CPU-vs-GPU consistency test (small mesh, 1 h simulated)

Same package, same coarse mesh, `legacy` (CPU) vs the selected `MODE`. Pass = wet area and water volume within 2 %.
A failure stops the notebook: do not trust GPU results then; use `MODE = "cpu"`.

In [ ]:
def run_one(dam, sid, mode, out, extra=(), gpu_id=None):
    env = dict(RUN_ENV)
    if gpu_id is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [sys.executable, "anuga_run.py", "--package", str(PKGS / dam), "--scenario", sid, "--mode", mode,
           "--out", str(out), *extra]
    with open(pathlib.Path(out).with_suffix(".log"), "w") as log:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            log.write(line)
            if line.startswith(("  t =", "mesh:", "scenario", "E1", "E3", "done", "Traceback", "RuntimeError")):
                print(f"[{dam}/{sid}/{mode}] {line.rstrip()}", flush=True)
        return p.wait()

if MODE != "legacy" and not SKIP_CONSISTENCY_TEST:
    d0 = DAMS[0]
    res = {}
    for m in ("legacy", MODE):
        o = WORK / "selftest" / m
        shutil.rmtree(o, ignore_errors=True)
        rc = run_one(d0, "selftest", m, o, ["--volume-m3", "5e6", "--max-area", "40000", "--finaltime-h", "1"])
        assert rc == 0, f"self-test {m} failed (see {o}.log)"
        res[m] = (json.loads((o / "stats.json").read_text())["inundated_km2"],
                  json.loads((o / "run.json").read_text())["final_volume_m3"],
                  json.loads((o / "run.json").read_text()).get("gpu_offload"))
    (a_w, a_v, _), (b_w, b_v, off) = res["legacy"], res[MODE]
    dw, dv = abs(b_w - a_w) / max(a_w, 1e-9), abs(b_v - a_v) / max(a_v, 1e-9)
    print(f"legacy: wet {a_w} km², volume {a_v:.0f} m³ | {MODE}: wet {b_w} km², volume {b_v:.0f} m³, gpu_offload={off}")
    print(f"relative difference: wet {dw * 100:.2f} %, volume {dv * 100:.3f} %")
    assert dw < 0.02 and dv < 0.02, "CPU vs GPU consistency test FAILED: do not use GPU results; set MODE = 'cpu'"
    print("consistency test passed")

## (e) Run the shard (resume-if-done) and (f) zip each run to the results folder

In [ ]:
todo = ALL if SHARD == "all" else [(d, s) for d, s in ALL if (d, s["id"]) in set(map(tuple, SHARD))]

def done(dam, sc):
    rj = RESULTS / dam / sc["id"] / "run.json"
    if not rj.exists():
        return False
    m = json.loads(rj.read_text())
    return m.get("simulated_s", 0) >= sc["finaltime_h"] * 3600 - 1 and (RESULTS / dam / sc["id"] / f"{dam}__{sc['id']}.zip").exists()

def job(dam, sc, gpu_id=None):
    out = WORK / "runs" / dam / sc["id"]
    shutil.rmtree(out, ignore_errors=True)
    out.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    rc = run_one(dam, sc["id"], MODE, out, gpu_id=gpu_id)
    dst = RESULTS / dam / sc["id"]
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(out.with_suffix(".log"), dst / "run.log")
    if rc != 0:
        print(f"!! {dam}/{sc['id']} failed (rc {rc}), log in {dst / 'run.log'}")
        return
    zp = dst / f"{dam}__{sc['id']}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in out.iterdir():
            if f.suffix != ".msh":
                zf.write(f, f"{dam}/{sc['id']}/{f.name}")
    for f in ("run.json", "tests.json", "stats.json"):
        shutil.copy(out / f, dst / f)
    print(f"== {dam}/{sc['id']} done in {(time.time() - t0) / 60:.1f} min -> {zp} ({zp.stat().st_size / 1e6:.1f} MB)")

pending = [(d, s) for d, s in todo if not done(d, s)]
print(f"{len(todo)} runs in shard, {len(todo) - len(pending)} already done, {len(pending)} to run")
ngpu = len(GPUS) if (MODE == "gpu" and PARALLEL_GPUS) else 1
if ngpu > 1:
    from concurrent.futures import ThreadPoolExecutor
    import queue
    slots = queue.Queue()
    for g in range(ngpu):
        slots.put(g)
    def worker(item):
        g = slots.get()
        try:
            job(*item, gpu_id=g)
        finally:
            slots.put(g)
    with ThreadPoolExecutor(ngpu) as ex:
        list(ex.map(worker, pending))
else:
    for i, (d, s) in enumerate(pending, 1):
        print(f"--- [{i}/{len(pending)}] {d} / {s['id']} (est. {s['est_triangles']} triangles)")
        job(d, s)

## Summary and download

Each run folder holds `<dam>__<scenario>.zip` + `run.json` / `tests.json` / `stats.json`. Bring the zips back and run
locally: `python pipeline/v2/anuga_ingest.py --results <colab results dir> <kaggle results dir>`. The ingest merges both,
reports missing or duplicated runs and computes the E4 mesh-sensitivity comparison. On Kaggle, also download
`/kaggle/working/wheels/` once and re-upload it as the `dwr-anuga-wheel` dataset.

In [ ]:
rows = []
for rj in sorted(RESULTS.glob("*/*/run.json")):
    m = json.loads(rj.read_text())
    t = json.loads((rj.parent / "tests.json").read_text()) if (rj.parent / "tests.json").exists() else {}
    rows.append((m["dam_id"], m["id"], m.get("mode"), m.get("gpu_offload"), m["triangles"], round(m["wall_s"] / 60, 1),
                 t.get("E1", {}).get("passed"), t.get("E2", {}).get("passed"), t.get("E3", {}).get("passed"),
                 t.get("E7", {}).get("passed")))
print("dam | scenario | mode | gpu_offload | triangles | wall min | E1 | E2 | E3 | E7")
for r in rows:
    print(" | ".join(map(str, r)))
if ON_KAGGLE:
    shutil.make_archive("/kaggle/working/results_all", "zip", RESULTS)
    print("download /kaggle/working/results_all.zip from the Output tab")